# 批量回测（两阶段：先缓存预测信号，再快速回测）

Phase 1 (慢，一次性的): 遍历所有模型，生成预测信号并保存到 `output/predictions_cache/`

Phase 2 (快): 从缓存加载预测信号，运行回测（不加载模型），收集结果

In [ ]:
import os
import glob
import pickle
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '../code/src')
sys.path.insert(0, '../code')
from backtest import ETFBacktester, run_backtest_from_predictions
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 配置参数

In [ ]:
YEAR = 2026
START_DATE = f"{YEAR}-04-01"
END_DATE = f"{YEAR}-05-12"
FIRST_REBALANCE_DATE = f"{YEAR}-04-01"
DATA_PATH = "../etf_data/etf_74.csv"
CACHE_DIR = "../output/predictions_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

TOP_K = 3
REBALANCE_DAYS = 5
POSITION_PCT = 0.95
INITIAL_CAPITAL = 100000
TRADE_MODE = "close"  # 或 "open"

# 模型目录
BASE_DIR = "../model"
MODEL_TYPES = [
    "bayes_itransformer_74_3",
    "search_itransformer_74_3",
    "bayes_dlinear_74_3",
    "bayes_lstm_74_3",
    "bayes_gru_74_3",
    "search_tcn_74_3",
]

# 收集所有实验
EXPERIMENTS = []
for MODEL_TYPE in MODEL_TYPES:
    for exp_dir in sorted(glob.glob(f"{BASE_DIR}/{MODEL_TYPE}/exp_*")):
        if os.path.exists(f"{exp_dir}/best_model_sliding.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_sliding.pth"))
        if os.path.exists(f"{exp_dir}/best_model.pth"):
            EXPERIMENTS.append((exp_dir, "best_model.pth"))
    cnt = len([e for e in EXPERIMENTS if MODEL_TYPE in e[0]])
    print(MODEL_TYPE, cnt)

print(f"实验总数: {len(EXPERIMENTS)}")

## Phase 1: 生成预测信号（加载模型，慢）

In [ ]:
import pickle

# 先加载一次数据（所有模型共享）
cached_data, cached_features = ETFBacktester.load_data_once(
    data_path=DATA_PATH,
    scaler_path=f"{EXPERIMENTS[0][0]}/scaler.pkl",
    feature_num="39",
    verbose=True,
)
print(f"数据加载完成: {cached_data['processed'].shape}")

In [ ]:
# Phase 1: 遍历所有模型，生成预测信号并缓存
CHECKPOINT_FILE = f"backtest_phase1_checkpoint_{YEAR}.pkl"

# 已完成的实验
completed = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'rb') as f:
        completed = pickle.load(f)
    print(f"加载断点: 已完成 {len(completed)} 个实验")

remaining = [(exp_dir, mf) for (exp_dir, mf) in EXPERIMENTS 
             if f"{exp_dir}/{mf}" not in completed]
print(f"剩余: {len(remaining)} 个实验")

for exp_dir, model_file in tqdm(remaining, desc="生成预测"):
    cache_key = f"{exp_dir}/{model_file}"
    safe_name = cache_key.replace("\\", "/").replace("../", "").replace("./", "").replace("/", "_")
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    try:
        # 使用缓存数据创建回测器
        bt = ETFBacktester.from_cached_data(
            model_dir=exp_dir,
            cached_data=cached_data,
            cached_features=cached_features,
            device='cpu',
            model_file=model_file,
            verbose=False,
        )
        
        # 生成预测
        preds_dict = bt.generate_predictions_dict(
            start_date=START_DATE,
            end_date=END_DATE,
        )
        
        # 保存到缓存

        with open(cache_path, 'wb') as f:
            pickle.dump(preds_dict, f)
        
        # 释放模型
        del bt.model
        del bt
        
        completed.add(cache_key)
        
    except Exception as e:
        print(f"\n失败: {cache_key}, 错误: {e}")
    
    # 每10个保存一次断点
    if len(completed) % 10 == 0:
        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump(completed, f)

# 最终保存
with open(CHECKPOINT_FILE, 'wb') as f:
    pickle.dump(completed, f)
print(f"\nPhase 1 完成，共 {len(completed)} 个实验")

## Phase 2: 从缓存加载预测信号，快速回测

In [ ]:
results = []
CHECKPOINT_RESULT = f"backtest_results_{YEAR}.pkl"

# 加载已有结果
if os.path.exists(CHECKPOINT_RESULT):
    with open(CHECKPOINT_RESULT, 'rb') as f:
        results = pickle.load(f)
    print(f"加载已有结果: {len(results)} 个实验")

completed_cache = set()
for r in results:
    completed_cache.add(r['cache_key'])

remaining = [(exp_dir, mf) for (exp_dir, mf) in EXPERIMENTS 
             if f"{exp_dir}/{mf}" not in completed_cache]
print(f"剩余: {len(remaining)} 个实验")

for exp_dir, model_file in tqdm(remaining, desc="回测"):
    cache_key = f"{exp_dir}/{model_file}"
    safe_name = cache_key.replace("\\", "/").replace("../", "").replace("./", "").replace("/", "_")
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    if not os.path.exists(cache_path):
        print(f"缓存不存在: {cache_path}")
        continue
    
    with open(cache_path, 'rb') as f:
        preds_dict = pickle.load(f)
    
    try:
        result = run_backtest_from_predictions(
            predictions_dict=preds_dict,
            data_path=DATA_PATH,
            start_date=START_DATE,
            end_date=END_DATE,
            top_k=TOP_K,
            rebalance_days=REBALANCE_DAYS,
            position_pct=POSITION_PCT,
            initial_capital=INITIAL_CAPITAL,
            commission=0.0003,
            first_rebalance_date=FIRST_REBALANCE_DATE,
            trade_mode=TRADE_MODE,
            verbose=False,
            log=False,
        )
        
        results.append({
            'cache_key': cache_key,
            'experiment': exp_dir.replace('\\', '/').split('/')[-2] + '/' + exp_dir.replace('\\', '/').split('/')[-1],
            'model_file': model_file,
            'strategy_return': result.strategy_return,
            'hs300_return': result.hs300_return,
            'excess_return': result.excess_return,
            'max_drawdown': result.max_drawdown,
            'drawdown_days': result.drawdown_days,
            'recovery_days': result.recovery_days,
            'recovered': result.recovered,
        })
        
    except Exception as e:
        print(f"\n失败: {cache_key}, 错误: {e}")
        results.append({
            'cache_key': cache_key,
            'experiment': exp_dir.replace('\\', '/').split('/')[-2] + '/' + exp_dir.replace('\\', '/').split('/')[-1],
            'model_file': model_file,
            'strategy_return': np.nan,
            'hs300_return': np.nan,
            'excess_return': np.nan,
            'max_drawdown': np.nan,
            'drawdown_days': np.nan,
            'recovery_days': np.nan,
            'recovered': np.nan,
        })
    
    # 每20个保存一次
    if len(results) % 20 == 0:
        with open(CHECKPOINT_RESULT, 'wb') as f:
            pickle.dump(results, f)

# 最终保存
with open(CHECKPOINT_RESULT, 'wb') as f:
    pickle.dump(results, f)

print(f"\nPhase 2 完成，共 {len(results)} 个实验")

## 结果分析

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df.dropna(subset=['strategy_return'])
df = df.sort_values('strategy_return', ascending=False).reset_index(drop=True)

print(f"有效实验: {len(df)} / {len(results)}")
print(f"\n{'='*80}")
print(f"回测模式: {'收盘交易' if TRADE_MODE == 'close' else '开盘交易'}")
print(f"回测期间: {START_DATE} ~ {END_DATE}")
print(f"{'='*80}")

# Top 20
print(f"\nTop 20 结果:")
print(f"{'排名':<4} {'实验':<35} {'模型文件':<25} {'收益':>8} {'HS300':>8} {'超额':>8} {'回撤':>8}")
print('-'*96)
for i, row in df.head(20).iterrows():
    print(f"{i+1:<4} {row['experiment']:<35} {row['model_file']:<25} {row['strategy_return']:>7.2f}% {row['hs300_return']:>7.2f}% {row['excess_return']:>7.2f}% {row['max_drawdown']:>7.2f}%")

# 按模型类型汇总
print(f"\n按模型类型统计:")
for mt in MODEL_TYPES:
    subset = df[df['experiment'].str.contains(mt)]
    if len(subset) > 0:
        best_idx = subset['strategy_return'].idxmax()
        best = subset.loc[best_idx]
        print(f"  {mt}: best={best['strategy_return']:.2f}% ({best['experiment']}/{best['model_file']}), "
              f"mean={subset['strategy_return'].mean():.2f}%, median={subset['strategy_return'].median():.2f}%, n={len(subset)}")

In [ ]:
# 切换交易模式重新回测（不需要重新生成预测）
# 只需重新运行 Phase 2 前修改 TRADE_MODE 即可
print("要切换模式，请修改上方 TRADE_MODE 变量后重新运行 Phase 2")
print("预测信号缓存可复用，无需重新加载模型")